In [9]:
import pandas as pd

df1 = pd.read_csv('C:/Users/금미/instacart_google.csv')
df1.tail()

,별점,내용
1995,4,Good delivery app but if you are not a member ...
1996,1,This company double charged me for my order. A...
1997,1,it sucks the app fails to feature the location...
1998,1,customer service is awful. you just talk to bo...
1999,3,this review is strictly about the app and not ...


In [10]:
df2 = pd.read_excel('C:/Users/금미/Desktop/[BDA] 파이썬 프젝/newpj/1_리뷰크롤링/instacart_reviews_리뷰데이터(전체).xlsx')
df2.tail()

,name,rating,date,review,thumbsUp,replyContent,version
995,scott nixon,2,2025-02-03 15:51:22,Their employees dont know how to shop for groc...,0,NaN,NaN
996,Jonah Cloud,1,2025-02-21 05:19:56,Dont use to order from restaurants. Warning! T...,0,NaN,NaN
997,Josh L,1,2025-02-09 05:31:04,I had to find out the hard way that they only ...,0,NaN,NaN
998,Wendy Barber,5,2025-02-28 06:06:56,"I have used Instacart for years now, and they ...",0,NaN,NaN
999,Erica Sheaffer,1,2025-01-01 22:39:54,"Customer service is a joke, my shopper cancele...",0,NaN,NaN


In [11]:
df2 = df2[['rating', 'review']]
df2

,rating,review
0,4,So far this app works great and Ive had no pro...
1,1,Horrible customer support. Items are often mis...
2,1,I dont know what is wrong with their billing s...
3,1,The delivery times you see before placing your...
4,2,Only order from this app because of the limite...
...,...,...
995,2,Their employees dont know how to shop for groc...
996,1,Dont use to order from restaurants. Warning! T...
997,1,I had to find out the hard way that they only ...
998,5,"I have used Instacart for years now, and they ..."


In [12]:
df2.rename(columns = {'rating' : '별점'}, inplace = True)
df2.rename(columns = {'review' : '내용'}, inplace = True)

df = pd.concat([df1, df2]).reset_index(drop=True)
df.tail()

,별점,내용
2995,2,Their employees dont know how to shop for groc...
2996,1,Dont use to order from restaurants. Warning! T...
2997,1,I had to find out the hard way that they only ...
2998,5,"I have used Instacart for years now, and they ..."
2999,1,"Customer service is a joke, my shopper cancele..."


In [13]:
df['별점'].value_counts()

별점
1    1344
5     751
2     336
4     289
3     280
Name: count, dtype: int64

# 텍스트 벡터화

In [24]:
import nltk

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\금미\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [25]:
import re
from nltk.tokenize import word_tokenize

# 전처리 함수: 소문자 변환, 구두점/이모지 제거, 토큰화
def preprocess_text(text):
    text = str(text).lower()  # 소문자 변환
    text = re.sub(r'[^a-z\s]', '', text)  # 알파벳과 공백 제외 제거
    tokens = word_tokenize(text)  # 토큰화
    return tokens

# '내용' 컬럼 전처리 적용
df['내용_토큰'] = df['내용'].apply(preprocess_text)

df['내용_토큰'].head()

0    [instacart, should, hold, the, shoppers, more,...
1    [the, delivery, times, you, see, before, placi...
2    [only, order, from, this, app, because, of, th...
3    [i, love, this, app, i, order, my, groceries, ...
4    [i, love, this, service, and, every, time, i, ...
Name: 내용_토큰, dtype: object

In [26]:
import nltk
from nltk.corpus import stopwords

# 불용어 다운로드 및 불용어 리스트 확보
nltk.download('stopwords')
stop_words_list = stopwords.words('english')
print('불용어 개수: ', len(stop_words_list))

# 노이즈 및 불용어 제거 함수
def clean_tokens(tokens):
    return [word for word in tokens if word not in stop_words_list]

# 내용_토큰 컬럼에서 정제 수행
df['내용_정제'] = df['내용_토큰'].apply(clean_tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\금미\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


불용어 개수:  198


In [27]:
from nltk.stem import WordNetLemmatizer
import nltk

# 표제어 추출용 리소스 다운로드 (최초 1회)
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

# 표제어 추출 함수
def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

# 컬럼 생성
df['표제어_추출'] = df['내용_정제'].apply(lemmatize_tokens)

df.head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\금미\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\금미\AppData\Roaming\nltk_data...


,별점,내용,내용_토큰,내용_정제,표제어_추출
0,2,Instacart should hold the shoppers more accoun...,"[instacart, should, hold, the, shoppers, more,...","[instacart, hold, shoppers, accountable, shopp...","[instacart, hold, shopper, accountable, shoppe..."
1,1,The delivery times you see before placing your...,"[the, delivery, times, you, see, before, placi...","[delivery, times, see, placing, order, nothing...","[delivery, time, see, placing, order, nothing,..."
2,2,Only order from this app because of the limite...,"[only, order, from, this, app, because, of, th...","[order, app, limited, options, grocery, delive...","[order, app, limited, option, grocery, deliver..."
3,5,"I love this app. I order my groceries, schedul...","[i, love, this, app, i, order, my, groceries, ...","[love, app, order, groceries, schedule, delive...","[love, app, order, grocery, schedule, delivery..."
4,5,I love this service and every time I have an i...,"[i, love, this, service, and, every, time, i, ...","[love, service, every, time, issue, rare, inst...","[love, service, every, time, issue, rare, inst..."


In [28]:
df[['내용_정제', '표제어_추출']]

,내용_정제,표제어_추출
0,"[instacart, hold, shoppers, accountable, shopp...","[instacart, hold, shopper, accountable, shoppe..."
1,"[delivery, times, see, placing, order, nothing...","[delivery, time, see, placing, order, nothing,..."
2,"[order, app, limited, options, grocery, delive...","[order, app, limited, option, grocery, deliver..."
3,"[love, app, order, groceries, schedule, delive...","[love, app, order, grocery, schedule, delivery..."
4,"[love, service, every, time, issue, rare, inst...","[love, service, every, time, issue, rare, inst..."
...,...,...
2995,"[employees, dont, know, shop, groceries, consi...","[employee, dont, know, shop, grocery, consiste..."
2996,"[dont, use, order, restaurants, warning, custo...","[dont, use, order, restaurant, warning, custom..."
2997,"[find, hard, way, like, one, shopper, area, tw...","[find, hard, way, like, one, shopper, area, tw..."
2998,"[used, instacart, years, always, good, option,...","[used, instacart, year, always, good, option, ..."


In [29]:
df.to_excel('0630_텍스트벡터화.xlsx', index=False)